# Final Project

## About the Project

This notebook explores the relationship between various socio-economic factors and happiness across different countries. Specifically, it analyzes data on income levels, internet users, GDP per capita, and happiness scores from the World Bank and World Happiness Report. The goal is to identify patterns, visualize trends, and understand how these indicators correlate with each other.

## Resources

*   [World Bank API Documentation](https://data.worldbank.org/developers/api-overview)
*   [World Happiness Report (Kaggle)](https://www.kaggle.com/datasets/mirichoi0218/world-happiness-report-2024)
*   [Plotly Python Graphing Library](https://plotly.com/python/)
*   [Pandas Documentation](https://pandas.pydata.org/docs/)
*   [Requests Library Documentation](https://requests.readthedocs.io/en/latest/)

### Credits
*   **World Bank API:** Data for GDP per capita, Internet Users, and Country Metadata.
*   **Kaggle:** World Happiness Report 2024 dataset.
*   **Plotly Documentation:** For guidance on creating interactive visualizations and choropleth maps.
*   **Stack Overflow:** For solutions to specific data manipulation and visualization challenges.
*   **ChatGPT:** For assistance with generating initial markdown narratives and refining code comments.

In [26]:
# Import necessary packages
import requests
import pandas as pd
import plotly.express as px
import plotly.io as pio

## Preparing data

This section focuses on preparing the raw data obtained from various sources. It involves making API requests, loading local datasets, selecting relevant columns, cleaning data types, and handling missing values to ensure the data is suitable for analysis and visualization.

First, I fetch country metadata from the World Bank API. This dataset provides essential information about countries, such as their region and income level, which is crucial for categorizing and analyzing other indicators. I extract the relevant columns and clean the 'region' and 'incomeLevel' fields, which are initially nested dictionaries, to plain string values. We also remove aggregate entries, as they do not represent individual countries.

In [27]:
# Make my first GET request: country metadata
countries_url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

response = requests.get(countries_url)
country_data = response.json()

countries = pd.DataFrame(country_data[1])
countries.head()

,id,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
0,ABW,AW,Aruba,"{'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Lati...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'HIC', 'iso2code': 'XD', 'value': 'High...","{'id': 'LNX', 'iso2code': 'XX', 'value': 'Not ...",Oranjestad,-70.0167,12.5167
1,AFE,ZH,Africa Eastern and Southern,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
2,AFG,AF,Afghanistan,"{'id': 'MEA', 'iso2code': 'ZQ', 'value': 'Midd...","{'id': 'MNA', 'iso2code': 'XQ', 'value': 'Midd...","{'id': 'LIC', 'iso2code': 'XM', 'value': 'Low ...","{'id': 'IDX', 'iso2code': 'XI', 'value': 'IDA'}",Kabul,69.1761,34.5228
3,AFR,A9,Africa,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,
4,AFW,ZI,Africa Western and Central,"{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': ''}","{'id': 'NA', 'iso2code': 'NA', 'value': 'Aggre...","{'id': '', 'iso2code': '', 'value': 'Aggregates'}",,,


In [28]:
# Select useful columns from the data
countries_df = countries[["id", "iso2Code", "name", "region", "incomeLevel"]].copy()

# The 'region' and 'incomeLevel' columns are stored as dictionaries
# We extract only the 'value' (the readable name)
countries_df["region"] = countries_df["region"].apply(
    lambda x: x["value"] if x is not None else None
)
countries_df["incomeLevel"] = countries_df["incomeLevel"].apply(
    lambda x: x["value"] if x is not None else None
)

# Remove aggregate entries because they are not individual countries.
countries_df = countries_df[countries_df["region"] != "Aggregates"]

countries_df.head()

,id,iso2Code,name,region,incomeLevel
0,ABW,AW,Aruba,Latin America & Caribbean,High income
2,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
5,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income
6,ALB,AL,Albania,Europe & Central Asia,Upper middle income
7,AND,AD,Andorra,Europe & Central Asia,High income


Next, I fetch data on internet users as a percentage of the population from the World Bank API. This dataset helps me understand the change of internet access across different countries over time. I select the country's ISO3 code, year, and the percentage of internet users. Then I rename the columns for clarity and convert 'year' and 'internet_users_pct' to numeric types.

In [29]:
# Make my second request: internet users
internet_url = "https://api.worldbank.org/v2/country/all/indicator/IT.NET.USER.ZS?format=json&per_page=20000"

response = requests.get(internet_url)
internet_data = response.json()

internet = pd.DataFrame(internet_data[1])
internet.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'IT.NET.USER.ZS', 'value': 'Individuals...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,30.4,,,0
1,"{'id': 'IT.NET.USER.ZS', 'value': 'Individuals...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,28.8,,,0
2,"{'id': 'IT.NET.USER.ZS', 'value': 'Individuals...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,27.8,,,0
3,"{'id': 'IT.NET.USER.ZS', 'value': 'Individuals...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,26.8,,,0
4,"{'id': 'IT.NET.USER.ZS', 'value': 'Individuals...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,25.0,,,0


In [30]:
# Select useful columns from the data
internet_df = internet[["countryiso3code", "date", "value"]].copy()
internet_df.columns = ["id", "year", "internet_users_pct"]

# Convert columns to numeric format
internet_df["year"] = pd.to_numeric(internet_df["year"], errors="coerce")
internet_df["internet_users_pct"] = pd.to_numeric(internet_df["internet_users_pct"], errors="coerce")

internet_df.head()

,id,year,internet_users_pct
0,AFE,2025,30.4
1,AFE,2024,28.8
2,AFE,2023,27.8
3,AFE,2022,26.8
4,AFE,2021,25.0


Similarly, I obtain GDP per capita data from the World Bank API. GDP per capita is a key economic indicator that helps me assess the economic development of each country. I extract the country's ISO3 code, year, and the GDP per capita value, renaming the columns and converting them to appropriate numeric types.

In [31]:
# Make my third request: GDP per capita
gdp_url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=20000"

response = requests.get(gdp_url)
gdp_data = response.json()

gdp = pd.DataFrame(gdp_data[1])
gdp.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN,,,1
1,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,1615.396356,,,1
2,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,1571.449189,,,1
3,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,1679.327622,,,1
4,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,1562.416175,,,1


In [32]:
# Select useful columns from the data
gdp_df = gdp[["countryiso3code", "date", "value"]].copy()
gdp_df.columns = ["id", "year", "gdp_per_capita"]

# Convert columns to numeric format
gdp_df["year"] = pd.to_numeric(gdp_df["year"], errors="coerce")
gdp_df["gdp_per_capita"] = pd.to_numeric(gdp_df["gdp_per_capita"], errors="coerce")

gdp_df.head()

,id,year,gdp_per_capita
0,AFE,2025,NaN
1,AFE,2024,1615.396356
2,AFE,2023,1571.449189
3,AFE,2022,1679.327622
4,AFE,2021,1562.416175


Finally, I download the World Happiness Report data for 2024 from a local CSV file. This dataset contains the 'Ladder score', which represents the happiness score for each country. I select only the 'Country name' and 'Ladder score' columns and rename them to 'name' and 'happiness_score' for consistency.

In [33]:
# Read the hapiness data (2024)
happiness_df = pd.read_csv("../data/World_Hapiness_Data.csv")
happiness_df.head()

,Country name,Ladder score,upperwhisker,lowerwhisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Finland,7.741,7.815,7.667,1.844,1.572,0.695,0.859,0.142,0.546,2.082
1,Denmark,7.583,7.665,7.500,1.908,1.520,0.699,0.823,0.204,0.548,1.881
2,Iceland,7.525,7.618,7.433,1.881,1.617,0.718,0.819,0.258,0.182,2.050
3,Sweden,7.344,7.422,7.267,1.878,1.501,0.724,0.838,0.221,0.524,1.658
4,Israel,7.341,7.405,7.277,1.803,1.513,0.740,0.641,0.153,0.193,2.298


In [34]:
# Clean the column names
happiness_df_clean = happiness_df[["Country name", "Ladder score"]].copy()

happiness_df_clean.columns = ["name", "happiness_score"]

happiness_df_clean.head()

,name,happiness_score
0,Finland,7.741
1,Denmark,7.583
2,Iceland,7.525
3,Sweden,7.344
4,Israel,7.341


After preparing individual datasets, the next step is to merge them into a single DataFrame. This allows me to analyze the relationships between internet access, GDP per capita, and happiness scores. I perform inner merges to ensure that only countries present in all datasets are included. I also remove any rows with missing values in critical columns to maintain data quality for subsequent analysis.

### Merge data

In [35]:
# Combine the datasets
merged = internet_df.merge(gdp_df, on=["id", "year"], how="inner")
merged = merged.merge(countries_df, on="id", how="left")

# Remove missing values
merged = merged.dropna(subset=["internet_users_pct", "gdp_per_capita", "region", "incomeLevel"])

merged.head()

,id,year,internet_users_pct,gdp_per_capita,iso2Code,name,region,incomeLevel
4556,AFG,2023,15.928400,413.757895,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4557,AFG,2022,15.866300,357.261153,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4558,AFG,2021,16.514299,356.496214,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4559,AFG,2020,17.048500,510.787063,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4560,AFG,2019,17.600000,496.602504,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income


## Visualization

This section focuses on visualizing the cleaned and merged data to uncover insights and patterns. I use Plotly to create interactive charts, including scatter plots and choropleth maps.

Before creating the interactive plot, I filtered the merged dataset to one recent year so that each country would appear only once in the scatterplot. Since some of the most recent (2025) rows contain missing GDP values, I selected the year of 2024 with complete enough data for both GDP per capita and internet usage.

In [36]:
# Find the latest year available in the merged dataset
merged["year"].max()

2024

In [37]:
# Create a dataset for one recent year (2024)
internet_2024 = merged[merged["year"] == 2024].copy()

# Preview the filtered data
internet_2024.head()

,id,year,internet_users_pct,gdp_per_capita,iso2Code,name,region,incomeLevel
4621,ALB,2024,85.863899,11377.775743,AL,Albania,Europe & Central Asia,Upper middle income
4687,DZA,2024,77.417801,5752.990767,DZ,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income
4819,AND,2024,94.397598,49303.649167,AD,Andorra,Europe & Central Asia,High income
4885,AGO,2024,40.704800,2665.874448,AO,Angola,Sub-Saharan Africa,Lower middle income
4951,ATG,2024,72.730400,23542.452695,AG,Antigua and Barbuda,Latin America & Caribbean,High income


### Interactive visualization: GDP per capita and internet access

To explore the relationship between economic development and technology access, I created an interactive scatterplot using Plotly. Each point represents a country. The x-axis shows GDP per capita on a log scale, and the y-axis shows the percentage of internet users. I also colored the points by income level so it is easier to compare patterns across groups, and hovering over a point shows the country name.

In [38]:
# Create an interactive scatterplot
fig_scatter = px.scatter(
    internet_2024,
    x="gdp_per_capita",
    y="internet_users_pct",
    color="incomeLevel",
    hover_name="name",
    log_x=True,
    title="Internet Access vs GDP per Capita (2024)",
    labels={
        "gdp_per_capita": "GDP per capita (log scale)",
        "internet_users_pct": "Internet users (% of population)",
        "incomeLevel": "Income level"
    }
)

fig_scatter.show()

#  This creates an HTML file with my chart
pio.write_html(fig_scatter, file="internet_gdp_interactive.html", auto_open=False, include_plotlyjs="cdn")

This interactive scatterplot shows a clear positive relationship between GDP per capita and internet access. In general, countries with higher GDP per capita tend to have a larger percentage of internet users. The pattern also shows differences by income level, with high-income countries clustering near very high internet access rates, while low-income countries tend to have lower access overall. This suggests that economic prosperity is a strong determinant of internet penetration.

### Interactive time trend: internet access over time by region

To better understand how internet access has changed over time, I created an interactive line plot showing average internet usage by region. This makes it easier to compare how different parts of the world have changed over time rather than only looking at one global average.

In [39]:
# Group data by year and region
region_time = (
    merged.groupby(["year", "region"])["internet_users_pct"]
    .mean()
    .reset_index()
)
region_time.head()

,year,region,internet_users_pct
0,1990,East Asia & Pacific,0.019048
1,1990,Europe & Central Asia,0.065561
2,1990,Latin America & Caribbean,0.000000
3,1990,"Middle East, North Africa, Afghanistan & Pakistan",0.005286
4,1990,North America,0.382000


In [40]:
# Create an interactive line plot
fig_time = px.line(
    region_time,
    x="year",
    y="internet_users_pct",
    color="region",
    title="Internet Access Over Time by Region",
    labels={
        "year": "Year",
        "internet_users_pct": "Average internet users (% of population)",
        "region": "Region"
    }
)

fig_time.show()

#  This creates an HTML file with my chart
pio.write_html(fig_time, file="time_trend_interactive.html", auto_open=False, include_plotlyjs="cdn")

This interactive line plot illustrates the average internet access trends over time for different regions. It clearly shows a global increase in internet usage across all regions, with some regions experiencing more rapid growth than others. High-income regions like North America and Europe & Central Asia had higher initial internet penetration, while other regions have shown significant catch-up. This visualization helps to understand the evolution of internet access disparities globally.

### Interactive Scatterplot: internet access and happiness across countries

To explore how internet access relates to quality of life, I created an interactive scatterplot comparing internet usage and happiness scores across countries. Each point represents a country, with the x-axis showing the percentage of internet users and the y-axis showing the happiness score. The points are also colored by income level to highlight differences across economic groups.

In [41]:
# Merge Happiness with Internet Data

happiness_internet = internet_2024.merge(
    happiness_df_clean,
    on="name",
    how="inner"
)

happiness_internet.head()

,id,year,internet_users_pct,gdp_per_capita,iso2Code,name,region,incomeLevel,happiness_score
0,ALB,2024,85.863899,11377.775743,AL,Albania,Europe & Central Asia,Upper middle income,5.304
1,DZA,2024,77.417801,5752.990767,DZ,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,5.364
2,ARG,2024,89.667152,13969.783660,AR,Argentina,Latin America & Caribbean,Upper middle income,6.188
3,ARM,2024,81.339302,8556.214070,AM,Armenia,Europe & Central Asia,Upper middle income,5.455
4,AUS,2024,96.131401,64603.985631,AU,Australia,East Asia & Pacific,High income,7.057


In [42]:
# Check how many countries matched
print("World Bank countries:", internet_2024.shape[0])
print("Matched countries:", happiness_internet.shape[0])

World Bank countries: 174
Matched countries: 122


In [43]:
# Create an interactive scatterplot
fig_happiness = px.scatter(
    happiness_internet,
    x="internet_users_pct",
    y="happiness_score",
    color="incomeLevel",
    hover_name="name",
    title="Internet Access vs Happiness Score (2024)",
    labels={
        "internet_users_pct": "Internet users (% of population)",
        "happiness_score": "Happiness score",
        "incomeLevel": "Income level"
    }
)

fig_happiness.show()

#  This creates an HTML file with my chart
pio.write_html(fig_happiness, file="happiness_interactive.html", auto_open=False, include_plotlyjs="cdn")

This visualization suggests a positive relationship between internet access and happiness. In general, countries with higher levels of internet usage tend to report higher happiness scores. This pattern is especially clear among high-income countries, which cluster in the upper-right area of the plot, indicating both high internet access and high happiness levels. On the other hand, low-income countries tend to have lower internet access and lower happiness scores, appearing in the lower-left region of the graph.

However, the relationship is not perfectly linear. Some countries with moderate internet access still show varying happiness levels, suggesting that other factors, such as income, health, social support, and political stability, also play important roles in determining happiness.

### Interactive Map: Internet access, Happiness, and GDP per capita across the world

To provide a geographical perspective, I use maps to visualize the distribution of happiness, internet access, and GDP per capita across the world. Each map uses color intensity to represent the value of the chosen indicator for each country. Hovering over a country reveals detailed information about its internet users, GDP per capita, and happiness score.

In [44]:
# Interactive map (Happiness)
fig_map = px.choropleth(
    happiness_internet,
    locations="id",
    color="happiness_score",
    hover_name="name",
    hover_data={
        "internet_users_pct": ":.1f",
        "gdp_per_capita": ":,.0f",
        "happiness_score": ":.2f"
    },
    labels={
        "internet_users_pct": "Internet users (% of population)",
        "gdp_per_capita": "GDP per capita (USD)",
        "happiness_score": "Happiness score"
    },
    color_continuous_scale="Plasma",
    title="Global Happiness Levels (2024)"
)

fig_map.show()

#  This creates an HTML file with my chart
# pio.write_html(fig_map, file="happiness_map_interactive.html", auto_open=False, include_plotlyjs="cdn")

In [45]:
# Interactive map (Internet Access)
fig_map = px.choropleth(
    happiness_internet,
    locations="id",
    color="internet_users_pct",
    hover_name="name",
    hover_data={
        "internet_users_pct": ":.1f",
        "gdp_per_capita": ":,.0f",
        "happiness_score": ":.2f"
    },
    labels={
        "internet_users_pct": "Internet users (% of population)",
        "gdp_per_capita": "GDP per capita (USD)",
        "happiness_score": "Happiness score"
    },
    color_continuous_scale="Plasma",
    title="Global Internet Users (2024)"
)

fig_map.show()

# Increase the width of the plot
fig_time.update_layout(
    width=1000,
    height=650,
    margin=dict(l=80, r=250, t=100, b=80),
    autosize=False
)

#  This creates an HTML file with my chart
# pio.write_html(fig_map, file="internet_access_map_interactive.html", auto_open=False, include_plotlyjs="cdn")

In [46]:
# Interactive map (GDP)
fig_map = px.choropleth(
    happiness_internet,
    locations="id",
    color="gdp_per_capita",
    hover_name="name",
    hover_data={
        "internet_users_pct": ":.1f",
        "gdp_per_capita": ":,.0f",
        "happiness_score": ":.2f"
    },
    labels={
        "internet_users_pct": "Internet users (% of population)",
        "gdp_per_capita": "GDP per capita (USD)",
        "happiness_score": "Happiness score"
    },
    color_continuous_scale="Plasma",
    title="GDP (2024)"
)

fig_map.show()

# Increase the width of the plot
fig_time.update_layout(
    width=1000,
    height=650,
    margin=dict(l=80, r=250, t=100, b=80),
    autosize=False
)

#  This creates an HTML file with my chart
# pio.write_html(fig_map, file="gdp_map_interactive.html", auto_open=False, include_plotlyjs="cdn")

Finally, I implement a switchable map to allow users to dynamically change the displayed metric (happiness, internet access, or GDP per capita) on a single world map. This enhances interactivity and facilitates direct comparison of these indicators across different countries within the same visualization context.

In [47]:
# Switchable Map
fig = px.choropleth(
    happiness_internet,
    locations="id",
    color="happiness_score",
    hover_name="name",
    hover_data={
        "internet_users_pct": ":.1f",
        "gdp_per_capita": ":,.0f",
        "happiness_score": ":.2f"
    },
    labels={
        "internet_users_pct": "Internet users (% of population)",
        "gdp_per_capita": "GDP per capita (USD)",
        "happiness_score": "Happiness score"
    },
    color_continuous_scale="Plasma",
    title="World Map (2024)"
)

fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    args=[
                        {"z": [happiness_internet["happiness_score"]]},
                        {
                            "title": "Global Happiness Levels (2024)",
                            "coloraxis.colorbar.title.text": "Happiness score"
                        }
                    ],
                    label="Happiness",
                    method="update"
                ),

                dict(
                    args=[
                        {"z": [happiness_internet["internet_users_pct"]]},
                        {
                            "title": "Internet Access (2024)",
                            "coloraxis.colorbar.title.text": "Internet users (%)"
                            }
                    ],
                    label="Internet Access",
                    method="update"
                ),

                dict(
                    args=[
                        {"z": [happiness_internet["gdp_per_capita"]]},
                        {
                            "title": "GDP per Capita (2024)",
                            "coloraxis.colorbar.title.text": "GDP per capita"
                            }
                    ],
                    label="GDP per Capita",
                    method="update"
                )
            ],
            direction="down",
            showactive=True,
            x=0.1,
            y=1.05
        )
    ]
)

fig.show()

#  This creates an HTML file with my chart
pio.write_html(fig, file="map_interactive.html", auto_open=False, include_plotlyjs="cdn")

## Conclusion and Reflection

### New Techniques Used
For this project, one new technique I used was making interactive visualizations with Plotly, especially using the updatemenus feature to create switchable maps. I learned how to use it by reading the Plotly documentation and trying different examples myself. I decided to use this feature because I wanted users to easily switch between happiness, internet access, and GDP per capita on the same world map and compare the patterns more interactively.

### Reflection on the Process
**What surprised me:** I was surprised by how much data the World Bank API had and how easy it was to access different kinds of information from many countries. It was also really interesting to immediately see patterns in the graphs, especially the relationship between GDP per capita and internet access. Seeing the trends visually helped me understand the data much better.

**What was challenging:** One of the hardest parts was matching country names and ISO codes between the World Bank dataset and the World Happiness Report dataset because some names were written differently or had missing values. I also had some trouble cleaning the data and making sure the datasets merged correctly. Another challenge was adjusting the Plotly graphs and maps so they looked clear and interactive without becoming too crowded or difficult to read.